# 02 — Prepare Datasets
Converts Pascal VOC XML annotations to YOLO format and builds the ResNet50 ImageFolder structure.

**Run once.** Outputs persist in Drive and are skipped automatically on re-runs.

In [ ]:
import os
FLAG = "/content/.deps_ok_karyotype"
if not os.path.exists(FLAG):
    !pip -q install --no-cache-dir --upgrade --force-reinstall \
        "numpy==2.0.2" "pandas==2.2.2" "opencv-python==4.10.0.84" \
        "ultralytics==8.*" "torch" "torchvision" "tqdm==4.*" "matplotlib==3.*"
    open(FLAG,"w").write("ok")
    raise SystemExit("✅ Restart runtime, then re-run.")
print("✅ Dependencies OK")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, random, shutil
from pathlib import Path

ROOT           = "/content/drive/MyDrive/54816"
OUT_DIR        = f"{ROOT}/e2e_outputs_run"
MULTI_IMG_DIR  = f"{ROOT}/24_chromosomes_object/JEPG"
MULTI_XML_DIR  = f"{ROOT}/24_chromosomes_object/annotations"
SINGLE_IMG_DIR = f"{ROOT}/single_chromosomes_object/JEPG"
SINGLE_XML_DIR = f"{ROOT}/single_chromosomes_object/annotations"
SEED           = 0
# Paper Section 2.2: 70% train / 20% val / 10% test
TRAIN_RATIO    = 0.70
VAL_RATIO      = 0.20
# Remaining 10% is held out as test set

os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)


In [ ]:
import xml.etree.ElementTree as ET
import cv2
from tqdm import tqdm

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}

def list_images_dir(img_dir):
    p = Path(img_dir)
    files = sorted([x for x in p.iterdir() if x.suffix.lower() in IMG_EXTS])
    return [f.name for f in files], {f.name: str(f) for f in files}

def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    W = int(size.findtext("width"))  if size is not None else None
    H = int(size.findtext("height")) if size is not None else None
    objs = []
    for obj in root.findall("object"):
        name = (obj.findtext("name") or "").strip()
        b = obj.find("bndbox")
        if not name or b is None:
            continue
        objs.append((name,
                     float(b.findtext("xmin")), float(b.findtext("ymin")),
                     float(b.findtext("xmax")), float(b.findtext("ymax"))))
    return W, H, objs

def build_label_map(xml_dir):
    labels = set()
    for xp in glob.glob(os.path.join(xml_dir, "*.xml")):
        _,_,objs = parse_voc(xp)
        for n,*_ in objs: labels.add(n)
    labels = sorted(labels)
    return labels, {lab:i for i,lab in enumerate(labels)}

def voc2yolo(xmin,ymin,xmax,ymax,W,H):
    xmin=max(0.0,min(xmin,W-1)); xmax=max(0.0,min(xmax,W-1))
    ymin=max(0.0,min(ymin,H-1)); ymax=max(0.0,min(ymax,H-1))
    bw=max(1e-6,xmax-xmin); bh=max(1e-6,ymax-ymin)
    cx=xmin+bw/2; cy=ymin+bh/2
    return cx/W, cy/H, bw/W, bh/H

def prepare_yolo_dataset(img_dir, xml_dir, out_root, label_to_id, one_class=False):
    for sp in ["train","val"]:
        os.makedirs(os.path.join(out_root,"images",sp), exist_ok=True)
        os.makedirs(os.path.join(out_root,"labels",sp), exist_ok=True)
    img_names, img_paths = list_images_dir(img_dir)
    xml_by_stem = {Path(x).stem:x for x in glob.glob(os.path.join(xml_dir,"*.xml"))}
    pairs = [(n,img_paths[n],xml_by_stem[Path(n).stem])
             for n in img_names if Path(n).stem in xml_by_stem]
    random.Random(SEED).shuffle(pairs)
    n = len(pairs)
    tr_end = int(n*TRAIN_RATIO)
    va_end  = tr_end + int(n*VAL_RATIO)
    # test set (pairs[va_end:]) is held out — not written to disk
    for sp, items in [("train",pairs[:tr_end]),("val",pairs[tr_end:va_end])]:
        for name,imgp,xmlp in tqdm(items, desc=f"VOC→YOLO {Path(out_root).name}:{sp}"):
            shutil.copy(imgp, os.path.join(out_root,"images",sp,name))
            W,H,objs = parse_voc(xmlp)
            if W is None or H is None:
                im=cv2.imread(imgp); H,W=im.shape[:2]
            lines=[]
            for lab,x1,y1,x2,y2 in objs:
                cid = 0 if one_class else label_to_id[lab]
                cx,cy,bw,bh = voc2yolo(x1,y1,x2,y2,W,H)
                lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            open(os.path.join(out_root,"labels",sp,f"{Path(name).stem}.txt"),"w").write("\n".join(lines))

def write_yaml(out_root, names):
    y = os.path.join(out_root,"data.yaml")
    with open(y,"w") as f:
        f.write(f"path: {out_root}\ntrain: images/train\nval: images/val\n")
        f.write(f"nc: {len(names)}\nnames:\n")
        for i,n in enumerate(names): f.write(f"  {i}: {n}\n")
    return y

multi_labels, _ = build_label_map(MULTI_XML_DIR)
json.dump(multi_labels, open(os.path.join(OUT_DIR,"multi_labels.json"),"w"), indent=2)
print("[INFO] Chromosome classes:", len(multi_labels), "→", multi_labels)

YOLO_MULTI_ROOT = os.path.join(OUT_DIR,"yolo_multiclass")
YOLO_ONE_ROOT   = os.path.join(OUT_DIR,"yolo_oneclass")

if not os.path.isdir(os.path.join(YOLO_MULTI_ROOT,"images","train")):
    prepare_yolo_dataset(MULTI_IMG_DIR, MULTI_XML_DIR, YOLO_MULTI_ROOT,
                         {lab:i for i,lab in enumerate(multi_labels)}, False)
else: print("[SKIP] yolo_multiclass already exists")

if not os.path.isdir(os.path.join(YOLO_ONE_ROOT,"images","train")):
    prepare_yolo_dataset(MULTI_IMG_DIR, MULTI_XML_DIR, YOLO_ONE_ROOT,
                         {lab:i for i,lab in enumerate(multi_labels)}, True)
else: print("[SKIP] yolo_oneclass already exists")

print("[DONE] yaml_multi:", write_yaml(YOLO_MULTI_ROOT, multi_labels))
print("[DONE] yaml_one  :", write_yaml(YOLO_ONE_ROOT, ["chromosomes"]))


In [ ]:
# Build ResNet50 ImageFolder (train/val) from single-chromosome dataset
from tqdm import tqdm

RESNET_DATA_ROOT = os.path.join(OUT_DIR,"resnet_imagefolder")
tr = os.path.join(RESNET_DATA_ROOT,"train")
va = os.path.join(RESNET_DATA_ROOT,"val")

if os.path.isdir(tr) and os.path.isdir(va):
    print("[SKIP] ResNet ImageFolder already exists")
else:
    os.makedirs(tr, exist_ok=True); os.makedirs(va, exist_ok=True)
    img_names, img_paths = list_images_dir(SINGLE_IMG_DIR)
    xml_by_stem = {Path(x).stem:x for x in glob.glob(os.path.join(SINGLE_XML_DIR,"*.xml"))}
    pairs = [(n,img_paths[n],xml_by_stem[Path(n).stem])
             for n in img_names if Path(n).stem in xml_by_stem]
    random.Random(SEED).shuffle(pairs)
    cut = int(len(pairs)*TRAIN_RATIO)

    def first_label(xmlp):
        root = ET.parse(xmlp).getroot()
        obj  = root.find("object")
        return ((obj.findtext("name") or "").strip() if obj is not None else None)

    for sp, items in [("train",pairs[:cut]),("val",pairs[cut:])]:
        root_out = tr if sp=="train" else va
        for name,imgp,xmlp in tqdm(items, desc=f"ImageFolder {sp}"):
            lab = first_label(xmlp)
            if not lab: continue
            os.makedirs(os.path.join(root_out,lab), exist_ok=True)
            shutil.copy(imgp, os.path.join(root_out,lab,name))

    print("[DONE] ResNet ImageFolder:", RESNET_DATA_ROOT)
